In [1]:
!pip install tensorflow

In [2]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
df = pd.read_csv("dice_com-job_us_sample.csv")

df.head()

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate,shift,site_name,skills,uniq_id
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago,Telecommuting not available|Travel not required,NaN,SEE BELOW,418ff92580b270ef4e7c14f0ddfc36b4
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago,Telecommuting not available|Travel not required,NaN,"linux/unix, network monitoring, incident respo...",8aec88cba08d53da65ab99cf20f6f9d9
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago,Telecommuting not available|Travel not required,NaN,"Enterprise Solutions Architecture, business in...",46baa1f69ac07779274bcd90b85d9a72
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago,Telecommuting not available|Travel not required,NaN,Please see job description,3941b2f206ae0f900c4fba4ac0b18719
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago,Telecommuting not available|Travel not required,NaN,"Configuration Management, Developer, Linux, Ma...",45efa1f6bc65acc32bbbb953a1ed13b7


In [4]:
print(df.columns)

Index(['advertiserurl', 'company', 'employmenttype_jobstatus',
       'jobdescription', 'jobid', 'joblocation_address', 'jobtitle',
       'postdate', 'shift', 'site_name', 'skills', 'uniq_id'],
      dtype='str')


In [5]:
df = df[['jobtitle', 'jobdescription']]
df.head()

,jobtitle,jobdescription
0,AUTOMATION TEST ENGINEER,Looking for Selenium engineers...must have sol...
1,Information Security Engineer,The University of Chicago has a rapidly growin...
2,Business Solutions Architect,"GalaxE.SolutionsEvery day, our solutions affec..."
3,"Java Developer (mid level)- FT- GREAT culture,...",Java DeveloperFull-time/direct-hireBolingbrook...
4,DevOps Engineer,Midtown based high tech firm has an immediate ...


In [6]:
df = df[['jobtitle','jobdescription']]

In [7]:
df = df.dropna()

print(df.shape)

(22000, 2)


In [8]:


top_jobs = df['jobtitle'].value_counts().head(20).index

df = df[df['jobtitle'].isin(top_jobs)]

print("Dataset Shape:", df.shape)

print("\nTop 20 Job Titles:")

print(df['jobtitle'].value_counts())

Dataset Shape: (1408, 2)

Top 20 Job Titles:
jobtitle
Java Developer                                 174
Project Manager                                145
Network Engineer                               128
Software Engineer                              118
Business Analyst                               117
.Net Developer                                  70
DevOps Engineer                                 60
Systems Engineer                                55
Systems Administrator                           54
Web Developer                                   53
Senior Software Engineer                        51
Technical Writer                                49
Business Systems Analyst                        46
Software Developer                              45
Android Developer                               42
Senior Network Engineer                         42
Data Analyst                                    41
Senior Java Developer                           41
Robert Half Technology Accou

In [9]:
X = df['jobdescription']
y = df['jobtitle']

In [10]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Total Classes:", len(label_encoder.classes_))

Total Classes: 20


In [11]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X)

sequences = tokenizer.texts_to_sequences(X)

In [12]:
max_len = 200

X_pad = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)

print(X_pad.shape)

(1408, 200)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [14]:
model = Sequential([
    Embedding(10000, 64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\navee\Desktop\Data_Scientist\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.1323 - loss: 2.9418 - val_accuracy: 0.1560 - val_loss: 2.8483
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1581 - loss: 2.8318 - val_accuracy: 0.2199 - val_loss: 2.7653
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.2336 - loss: 2.7524 - val_accuracy: 0.2340 - val_loss: 2.6916
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.2336 - loss: 2.6452 - val_accuracy: 0.2801 - val_loss: 2.5730
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2957 - loss: 2.4617 - val_accuracy: 0.3156 - val_loss: 2.3803
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.3162 - loss: 2.2387 - val_accuracy: 0.3333 - val_loss: 2.1759
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3357 - loss: 2.0419 - val_accuracy: 0.3369 - val_loss: 2.0321
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3792 - loss: 1.8748 - val_accuracy: 0.3759 - v

In [16]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc * 100)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5816 - loss: 1.2969
Test Accuracy: 58.156025409698486


In [17]:
model.save("job_model.h5") 

print("Model Saved Successfully")

Model Saved Successfully


In [18]:
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

print("Tokenizer Saved")

Tokenizer Saved


In [19]:
with open("label_encoder.pkl","wb") as f:
    pickle.dump(label_encoder,f)

print("Label Encoder Saved")

Label Encoder Saved


In [20]:
sample = """
Python developer with machine learning,
SQL, Pandas, NumPy and Deep Learning skills
"""

seq = tokenizer.texts_to_sequences([sample])

pad = pad_sequences(seq,maxlen=max_len,padding='post')

pred = model.predict(pad)

job = label_encoder.inverse_transform([np.argmax(pred)])

print("Recommended Job:", job[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Recommended Job: Java Developer
